# Use several reservoirs or your own catchment

## Goal
FE Walter and Beltzville join on Lehigh contributions. Prompton drains through Lackawaxen contributions. These reservoirs are tributary branches, not a three-dam cascade. The managed outlet sums these contributions; it is not full river discharge. A true cascade connects an upstream outlet to the next reservoir inflow node, including local inflow and travel delay.

## Setup
Python 3.10+ with repository dependencies. All cells use local files; no provider calls.

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: clean-upgrade-99ea43e42a


## Steps

### Check and run 31 days with passive Prompton

In [2]:
settings, verified=ensure_offline('tutorials/config/multi_reservoir.offline.yaml')
print('Agents:',settings.report['agent_reservoirs'])
print('Passive count:',settings.report['passive_reservoir_count'])
print('Connections:',settings.network['connections'])
print(json.dumps(verified,indent=2))


Agents: {'fe_operator': 'FEWalter', 'belt_operator': 'Beltzville'}
Passive count: 1
Connections: [{'source': 'FEWalter', 'target': 'LehighContribution', 'unit_hydrograph': [1.0]}, {'source': 'Beltzville', 'target': 'LehighContribution', 'unit_hydrograph': [1.0]}, {'source': 'Prompton', 'target': 'LackawaxenContribution', 'unit_hydrograph': [1.0]}, {'source': 'LehighContribution', 'target': 'ManagedContributionOutlet', 'unit_hydrograph': [1.0]}, {'source': 'LackawaxenContribution', 'target': 'ManagedContributionOutlet', 'unit_hydrograph': [1.0]}]
{
  "status": "PASS",
  "days": 31,
  "reservoir_count": 3,
  "reservoir_days": 93,
  "date_start": "2016-01-01",
  "date_end": "2016-01-31",
  "audit_counts": {
    "PASS": 62,
    "UNKNOWN": 31
  },
  "max_reservoir_balance_residual_m3": 3.026798367500305e-09,
  "max_system_balance_residual_m3": 3.725290298461914e-09,
  "max_routing_difference_cms": 0.0,
  "final_channel_storage_m3": 0.0,
  "observed_network_calls": 0,
  "operating_strategy_

### Inspect routed contributions

In [3]:
import csv
rows=list(csv.DictReader((Path(settings.report['output'])/'network_flows.csv').open()))
print('Flow columns:', list(rows[0]))
print('First and last days:', rows[0], rows[-1])
print('Custom checklist: case geometry/units -> physics/profile -> own prompt/DKP -> auditor -> offline checks -> live readiness')

Flow columns: ['date', 'node', 'local_inflow_cms', 'routed_flow_cms']
First and last days: {'date': '2016-01-01', 'node': 'FEWalter_Inflow', 'local_inflow_cms': '32.12531870666853', 'routed_flow_cms': '32.12531870666853'} {'date': '2016-01-31', 'node': 'ManagedContributionOutlet', 'local_inflow_cms': '0.0', 'routed_flow_cms': '8.732709105737518'}
Custom checklist: case geometry/units -> physics/profile -> own prompt/DKP -> auditor -> offline checks -> live readiness


## Checks
Review the printed inputs, counts, source labels and residuals. Synthetic replies validate interfaces only.

## Next steps
UH=[1] is transparent connectivity, not calibrated downstream travel time. The tests also cover delayed routing, serial reservoirs, confluence and channel storage. Prompton strategy audit remains UNKNOWN; offline plumbing acceptance does not clear live-readiness blockers.